# Merged: FLAN-T5 Base Inspection + Training (Kaggle / T4, Internet On)

This notebook merges `flan-t5-base-inspection.ipynb` and `training.ipynb` into one runnable flow.
Core logic from both notebooks is preserved; only redundant/duplicate code paths were consolidated or renamed to avoid collisions.

In [145]:
# Optional: install/upgrade dependencies (Kaggle usually already has these)
INSTALL_DEPS = False
if INSTALL_DEPS:
    %pip -q install --upgrade transformers datasets sentencepiece accelerate

In [146]:
import re
import math
from collections import defaultdict, Counter

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import DataCollatorForSeq2Seq

In [147]:
# Execution switches (keep heavy sections off by default for Kaggle run stability)
RUN_MODEL_DEMOS = True
RUN_MODEL_INSPECTION_PRINTS = True
RUN_FULL_FFN_TRAINABILITY_DEMO = True
RUN_TINY_DEMO_TRAINING = True
RUN_LORA_DUMMY_TRAINING = True

# Keep dataset print/debug cells optional
RUN_DATASET_INSPECTION = True

# PubMedQA sections (these are the heaviest)
RUN_PUBMEDQA_PIPELINE_SIMPLE = True
RUN_PUBMEDQA_TRAINING_WITH_ACCURACY = True

# 1. Model Loading

In [148]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
).to(device)

print("Model loaded on:", device)

Model loaded on: cuda


In [149]:
if RUN_MODEL_DEMOS:
    model.eval()

    prompt = "What can you do"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128
        )

    print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

Make a sand castle


In [150]:
if RUN_MODEL_DEMOS:
    sample_input = "Question: Is water wet?? Instruction: Answer in one sentence."
    inputs = tokenizer(sample_input, return_tensors="pt").to(device)

    outputs = model.generate(**inputs, max_new_tokens=64)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

No


# 2. Model Inspection

In [151]:
if RUN_MODEL_INSPECTION_PRINTS:
    print(model)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [152]:
if RUN_MODEL_INSPECTION_PRINTS:
    encoder_block_0 = model.encoder.block[0]
    print(encoder_block_0)

T5Block(
  (layer): ModuleList(
    (0): T5LayerSelfAttention(
      (SelfAttention): T5Attention(
        (q): Linear(in_features=768, out_features=768, bias=False)
        (k): Linear(in_features=768, out_features=768, bias=False)
        (v): Linear(in_features=768, out_features=768, bias=False)
        (o): Linear(in_features=768, out_features=768, bias=False)
        (relative_attention_bias): Embedding(32, 12)
      )
      (layer_norm): T5LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): T5LayerFF(
      (DenseReluDense): T5DenseGatedActDense(
        (wi_0): Linear(in_features=768, out_features=2048, bias=False)
        (wi_1): Linear(in_features=768, out_features=2048, bias=False)
        (wo): Linear(in_features=2048, out_features=768, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
        (act): NewGELUActivation()
      )
      (layer_norm): T5LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
)


In [153]:
if RUN_MODEL_INSPECTION_PRINTS:
    decoder_block_0 = model.decoder.block[0]
    print(decoder_block_0)

T5Block(
  (layer): ModuleList(
    (0): T5LayerSelfAttention(
      (SelfAttention): T5Attention(
        (q): Linear(in_features=768, out_features=768, bias=False)
        (k): Linear(in_features=768, out_features=768, bias=False)
        (v): Linear(in_features=768, out_features=768, bias=False)
        (o): Linear(in_features=768, out_features=768, bias=False)
        (relative_attention_bias): Embedding(32, 12)
      )
      (layer_norm): T5LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): T5LayerCrossAttention(
      (EncDecAttention): T5Attention(
        (q): Linear(in_features=768, out_features=768, bias=False)
        (k): Linear(in_features=768, out_features=768, bias=False)
        (v): Linear(in_features=768, out_features=768, bias=False)
        (o): Linear(in_features=768, out_features=768, bias=False)
      )
      (layer_norm): T5LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (2): T5LayerFF(
      (DenseReluDense): T5DenseG

In [154]:
if RUN_MODEL_INSPECTION_PRINTS:
    lm_head = model.lm_head
    print(lm_head)

Linear(in_features=768, out_features=32128, bias=False)


In [155]:
if RUN_MODEL_INSPECTION_PRINTS:
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params

    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters:    {frozen_params:,}")

Total parameters: 247,577,856
Trainable parameters: 247,577,856
Frozen parameters:    0


# 3. Full Training (FFN-only trainability demo + tiny demo training)

In [156]:
def freeze_all_params(model):
    for p in model.parameters():
        p.requires_grad = True


def enable_ffn_training(model):
    """
    Enable training only for FFN (DenseReluDense) layers
    in both encoder and decoder.
    """
    for name, module in model.named_modules():
        if module.__class__.__name__ == "T5DenseGatedActDense":
            for p in module.parameters():
                p.requires_grad = True


def print_trainable_params(model):
    for name, p in model.named_parameters():
        if p.requires_grad:
            print(name)


def count_parameters(model):
    total = 0
    trainable = 0
    for p in model.parameters():
        n = p.numel()
        total += n
        if p.requires_grad:
            trainable += n
    return total, trainable

In [157]:
if RUN_FULL_FFN_TRAINABILITY_DEMO:
    param_stats = defaultdict(int)

    for name, param in model.named_parameters():
        param_stats[name.split('.')[0]] += param.numel()

    for k, v in param_stats.items():
        print(f"{k}: {v:,}")

shared: 24,674,304
encoder: 84,954,240
decoder: 113,275,008
lm_head: 24,674,304


In [158]:
if RUN_FULL_FFN_TRAINABILITY_DEMO:
    for name, param in model.named_parameters():
        if "DenseReluDense" in name:
            print("FFN:", name, param.numel())
        elif "SelfAttention" in name or "EncDecAttention" in name:
            print("ATTN:", name, param.numel())

ATTN: encoder.block.0.layer.0.SelfAttention.q.weight 589824
ATTN: encoder.block.0.layer.0.SelfAttention.k.weight 589824
ATTN: encoder.block.0.layer.0.SelfAttention.v.weight 589824
ATTN: encoder.block.0.layer.0.SelfAttention.o.weight 589824
ATTN: encoder.block.0.layer.0.SelfAttention.relative_attention_bias.weight 384
FFN: encoder.block.0.layer.1.DenseReluDense.wi_0.weight 1572864
FFN: encoder.block.0.layer.1.DenseReluDense.wi_1.weight 1572864
FFN: encoder.block.0.layer.1.DenseReluDense.wo.weight 1572864
ATTN: encoder.block.1.layer.0.SelfAttention.q.weight 589824
ATTN: encoder.block.1.layer.0.SelfAttention.k.weight 589824
ATTN: encoder.block.1.layer.0.SelfAttention.v.weight 589824
ATTN: encoder.block.1.layer.0.SelfAttention.o.weight 589824
FFN: encoder.block.1.layer.1.DenseReluDense.wi_0.weight 1572864
FFN: encoder.block.1.layer.1.DenseReluDense.wi_1.weight 1572864
FFN: encoder.block.1.layer.1.DenseReluDense.wo.weight 1572864
ATTN: encoder.block.2.layer.0.SelfAttention.q.weight 589824
A

In [159]:
if RUN_FULL_FFN_TRAINABILITY_DEMO:
    freeze_all_params(model)
    enable_ffn_training(model)
    print(sum(p.requires_grad for p in model.parameters()))

    print_trainable_params(model)

    total_params, trainable_params = count_parameters(model)
    percent = 100 * trainable_params / total_params

    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Trainable %: {percent:.2f}%")

282
shared.weight
encoder.block.0.layer.0.SelfAttention.q.weight
encoder.block.0.layer.0.SelfAttention.k.weight
encoder.block.0.layer.0.SelfAttention.v.weight
encoder.block.0.layer.0.SelfAttention.o.weight
encoder.block.0.layer.0.SelfAttention.relative_attention_bias.weight
encoder.block.0.layer.0.layer_norm.weight
encoder.block.0.layer.1.DenseReluDense.wi_0.weight
encoder.block.0.layer.1.DenseReluDense.wi_1.weight
encoder.block.0.layer.1.DenseReluDense.wo.weight
encoder.block.0.layer.1.layer_norm.weight
encoder.block.1.layer.0.SelfAttention.q.weight
encoder.block.1.layer.0.SelfAttention.k.weight
encoder.block.1.layer.0.SelfAttention.v.weight
encoder.block.1.layer.0.SelfAttention.o.weight
encoder.block.1.layer.0.layer_norm.weight
encoder.block.1.layer.1.DenseReluDense.wi_0.weight
encoder.block.1.layer.1.DenseReluDense.wi_1.weight
encoder.block.1.layer.1.DenseReluDense.wo.weight
encoder.block.1.layer.1.layer_norm.weight
encoder.block.2.layer.0.SelfAttention.q.weight
encoder.block.2.laye

In [160]:
if RUN_FULL_FFN_TRAINABILITY_DEMO:
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params

    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters:    {frozen_params:,}")

Trainable parameters: 247,577,856
Frozen parameters:    0


In [161]:
if RUN_FULL_FFN_TRAINABILITY_DEMO:
    sample_input = "Question: Does aspirin reduce heart attack risk? Instruction: Answer in 2 sentences."
    inputs = tokenizer(sample_input, return_tensors="pt").to(device)

    outputs = model.generate(**inputs, max_new_tokens=64)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Aspirin is a sedative that can reduce the risk of heart attack.


## Demo training

In [162]:
if RUN_TINY_DEMO_TRAINING:
    data = [
        {
            "instruction": "Question: Does aspirin reduce fever? Answer yes or no.",
            "output": "Yes, aspirin can reduce fever."
        },
        {
            "instruction": "Question: Is vitamin C a cure for cancer? Answer yes or no.",
            "output": "No, vitamin C is not a cure for cancer."
        }
    ]

In [163]:
class SimpleT5Dataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        enc = self.tokenizer(
            item["instruction"],
            truncation=True,
            padding=False
        )

        with self.tokenizer.as_target_tokenizer():
            labels = self.tokenizer(
                item["output"],
                truncation=True,
                padding=False,
            )["input_ids"]

        labels = [
            l if l != self.tokenizer.pad_token_id else -100
            for l in labels
        ]

        return {
            "input_ids": torch.tensor(enc["input_ids"]),
            "attention_mask": torch.tensor(enc["attention_mask"]),
            "labels": torch.tensor(labels)
        }

In [164]:
if RUN_TINY_DEMO_TRAINING:
    demo_dataset = SimpleT5Dataset(data, tokenizer)

    demo_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        label_pad_token_id=-100
    )

    demo_train_loader = DataLoader(
        demo_dataset,
        batch_size=2,
        shuffle=True,
        collate_fn=demo_collator
    )

In [165]:
if RUN_TINY_DEMO_TRAINING:
    model = model.float().to(device)

    from torch.optim import AdamW

    demo_optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-5
    )

In [166]:
if RUN_TINY_DEMO_TRAINING:
    num_epochs = 3

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        print(f"\nEpoch {epoch + 1}/{num_epochs}")

        for step, batch in enumerate(demo_train_loader, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}

            demo_optimizer.zero_grad()

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )

            loss = outputs.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                max_norm=1.0
            )

            demo_optimizer.step()

            total_loss += loss.item()
            print(f"Step {step} | Loss: {loss.item():.4f}")

        avg_loss = total_loss / len(demo_train_loader)
        print(f"Epoch {epoch + 1} average loss: {avg_loss:.4f}")


Epoch 1/3
Step 1 | Loss: 3.0790
Epoch 1 average loss: 3.0790

Epoch 2/3
Step 1 | Loss: 2.2980
Epoch 2 average loss: 2.2980

Epoch 3/3
Step 1 | Loss: 2.6609
Epoch 3 average loss: 2.6609


# 4. LoRA Training

## Apply LoRA to FFNs

In [167]:
def apply_lora_to_ffn(model, r=8, alpha=1.0):
    """
    Wrap the FFN linear layers (wi_0, wi_1, wo) inside each T5DenseGatedActDense
    with LoRA adapters.

    This function is idempotent: re-running it will not double-wrap layers that are
    already LoRA-wrapped.
    """
    for module in model.modules():
        if module.__class__.__name__ != "T5DenseGatedActDense":
            continue
        for attr in ("wi_0", "wi_1", "wo"):
            child = getattr(module, attr)
            if isinstance(child, LoRALinear):
                continue
            if not isinstance(child, nn.Linear):
                raise TypeError(f"Expected nn.Linear at {module.__class__.__name__}.{attr}, got {type(child)}")
            setattr(module, attr, LoRALinear(child, r=r, alpha=alpha))


def merge_lora_to_linear(model):
    """
    Replace all LoRALinear modules in the model with plain nn.Linear modules whose
    weights include the merged LoRA contribution.

    This walks the module tree and replaces children in their parent module.
    """
    def _merge_in_parent(parent: nn.Module):
        for name, child in list(parent._modules.items()):
            if isinstance(child, LoRALinear):
                base = child.base
                new_linear = nn.Linear(
                    base.in_features,
                    base.out_features,
                    bias=(base.bias is not None),
                )
                new_linear = new_linear.to(base.weight.device, dtype=base.weight.dtype)
                new_linear.weight.data = base.weight.data + (child.B.weight @ child.A.weight) * child.scaling
                if base.bias is not None:
                    new_linear.bias.data = base.bias.data
                parent._modules[name] = new_linear
            else:
                _merge_in_parent(child)

    _merge_in_parent(model)


class LoRALinear(nn.Module):
    def __init__(self, base_linear, r=8, alpha=1.0):
        super().__init__()
        # If this function is re-run on a model that already has LoRA, unwrap to the
        # underlying nn.Linear to avoid nested LoRA wrappers.
        while isinstance(base_linear, LoRALinear):
            base_linear = base_linear.base
        if not isinstance(base_linear, nn.Linear):
            raise TypeError(f"LoRALinear expects an nn.Linear base, got {type(base_linear)}")

        self.base = base_linear
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False

        in_dim = self.base.in_features
        out_dim = self.base.out_features

        self.r = int(r)
        self.alpha = float(alpha)
        self.scaling = self.alpha / self.r

        self.A = nn.Linear(in_dim, self.r, bias=False)
        self.B = nn.Linear(self.r, out_dim, bias=False)
        self.A = self.A.to(self.base.weight.device, dtype=self.base.weight.dtype)
        self.B = self.B.to(self.base.weight.device, dtype=self.base.weight.dtype)

        # Initialize LoRA
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.base(x) + self.scaling * self.B(self.A(x))

    # Expose weight/bias/in_features/out_features to satisfy some HF expectations
    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    @property
    def in_features(self):
        return self.base.in_features

    @property
    def out_features(self):
        return self.base.out_features


# Temporarily compute effective weights for generation
def enable_lora_forward(model):
    for module in model.modules():
        if isinstance(module, LoRALinear) and not hasattr(module, "_original_forward"):
            module._original_forward = module.forward
            module.forward = lambda x, m=module: m.base(x) + m.scaling * m.B(m.A(x))


# Restore original forward for training
def disable_lora_forward(model):
    for module in model.modules():
        if isinstance(module, LoRALinear) and hasattr(module, "_original_forward"):
            module.forward = module._original_forward
            del module._original_forward

## Dummy Training

In [168]:
if RUN_LORA_DUMMY_TRAINING:
    rank = 16
    alpha = 32

    freeze_all_params(model)
    apply_lora_to_ffn(model, r=rank, alpha=alpha)

    for name, param in model.named_parameters():
        if "A.weight" in name or "B.weight" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False

In [169]:
if RUN_LORA_DUMMY_TRAINING:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable

    print(f"Total parameters: {total:,}")
    print(f"Trainable parameters (LoRA): {trainable:,}")
    print(f"Frozen parameters: {frozen:,}")
    print(f"Trainable %: {100 * trainable / total:.4f}%")

Total parameters: 250,821,888
Trainable parameters (LoRA): 3,244,032
Frozen parameters: 247,577,856
Trainable %: 1.2934%


In [170]:
if RUN_LORA_DUMMY_TRAINING:
    class DummyDataset(Dataset):
        def __init__(self, tokenizer):
            self.data = [
                {"input": "Question: Is the sky blue? Instruction: Answer in one sentence.",
                 "output": "Yes, the sky is blue."},
                {"input": "Question: Do cats bark? Instruction: Answer in one sentence.",
                 "output": "No, cats do not bark."},
                {"input": "Question: Is water wet? Instruction: Answer in one sentence.",
                 "output": "Yes, water is wet."}
            ]
            self.tokenizer = tokenizer

        def __len__(self):
            return len(self.data)

        def __getitem__(self, idx):
            example = self.data[idx]
            input_enc = self.tokenizer(example["input"], return_tensors="pt", truncation=True, padding=False)
            with self.tokenizer.as_target_tokenizer():
                labels = self.tokenizer(example["output"], return_tensors="pt", truncation=True, padding=False)["input_ids"]
            labels[labels == self.tokenizer.pad_token_id] = -100

            return {
                "input_ids": input_enc["input_ids"].squeeze(0),
                "attention_mask": input_enc["attention_mask"].squeeze(0),
                "labels": labels.squeeze(0)
            }

    dummy_train_dataset = DummyDataset(tokenizer)
    dummy_train_loader = DataLoader(dummy_train_dataset, batch_size=1, shuffle=True)

In [171]:
if RUN_LORA_DUMMY_TRAINING:
    dummy_optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    model.to(device)
    model.train()

    num_epochs = 3

    for epoch in range(num_epochs):
        running_loss = 0.0
        for step, batch in enumerate(dummy_train_loader, 1):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            dummy_optimizer.zero_grad()
            loss.backward()
            dummy_optimizer.step()

            running_loss += loss.item()
            print(f"Step {step} | Loss: {loss.item():.4f}", end="\r")

        avg_loss = running_loss / len(dummy_train_loader)
        print(f"\nEpoch {epoch+1} average loss: {avg_loss:.4f}")

Step 3 | Loss: 1.4850
Epoch 1 average loss: 1.1625
Step 3 | Loss: 1.2944
Epoch 2 average loss: 1.0279
Step 3 | Loss: 0.6395
Epoch 3 average loss: 0.5635


## Inference after training

In [172]:
if RUN_LORA_DUMMY_TRAINING:
    merge_lora_to_linear(model)  # Always merge before inference

    model.to(device)
    model.eval()

    sample_input = "Question: Does aspirin reduce heart attack risk? Instruction: Answer in 2 sentences."
    inputs = tokenizer(sample_input, return_tensors="pt").to(device)

    outputs = model.generate(**inputs, max_new_tokens=64)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Aspirin is a drug that is used to treat heart attack.


In [173]:
if RUN_LORA_DUMMY_TRAINING:
    sample_input = "Question: Is water wet?? Instruction: Answer in one sentence."
    inputs = tokenizer(sample_input, return_tensors="pt").to(device)

    outputs = model.generate(**inputs, max_new_tokens=64)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Yes, water is wet.


# Training using PubMedQA DS

In [174]:
# Variant A (from inspection notebook)
def preprocess_pubmedqa_basic(example):
    context = " ".join(example["context"]["contexts"])

    input_text = (
        f"Question: {example['question']} "
        f"Context: {context} "
        f"Instruction: First answer yes, no, or maybe. "
        f"Then justify your answer briefly."
    )

    target_text = (
        f"Answer: {example['final_decision']}. "
        f"Explanation: {example['long_answer']}"
    )

    return {
        "input": input_text,
        "output": target_text,
    }



# Variant B (from training notebook; keeps short_answer for evaluation)
def preprocess_pubmedqa_with_short_answer(example):
    context = " ".join(example["context"]["contexts"])

    input_text = (
        f"Question: {example['question']} "
        f"Context: {context} "
        f"Instruction: Answer yes, no, or maybe. Then justify your answer."
    )

    target_text = (
        f"Answer: {example['final_decision']}. "
        f"Explanation: {example['long_answer']}"
    )
    short_answer = example["final_decision"]

    return {
        "input": input_text,
        "output": target_text,
        "short_answer": short_answer,
    }



# Memory-friendly defaults (dynamic padding; avoid padding everything to 512 tokens)
max_input_length = 256
max_output_length = 96
max_target_length = max_output_length  # compatibility alias


def tokenize_for_t5(example):
    # Encode inputs (no padding here; pad dynamically in collator)
    input_enc = tokenizer(
        example["input"],
        truncation=True,
        padding=False,
        max_length=max_input_length,
    )

    # Encode targets
    with tokenizer.as_target_tokenizer():
        target_enc = tokenizer(
            example["output"],
            truncation=True,
            padding=False,
            max_length=max_output_length,
        )

    labels = target_enc["input_ids"]
    return {
        "input_ids": input_enc["input_ids"],
        "attention_mask": input_enc["attention_mask"],
        "labels": labels,
    }


def collate_fn(batch):
    # Use HF collator to pad dynamically and apply -100 to label padding.
    # This reduces VRAM significantly compared to padding every sample to max_length.
    pad_to_multiple_of = 8 if torch.cuda.is_available() else None
    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        label_pad_token_id=-100,
        pad_to_multiple_of=pad_to_multiple_of,
    )
    return collator(batch)

## PubMedQA pipeline (simple)

In [175]:
if RUN_PUBMEDQA_PIPELINE_SIMPLE:
    dataset = load_dataset("pubmed_qa", "pqa_labeled")
    if RUN_DATASET_INSPECTION:
        print(dataset)
        print(dataset.column_names)
        print(dataset["train"].features)
        print(dataset["train"][0])
        print(dataset["train"][0]["final_decision"])
    print(set(dataset["train"]["final_decision"]))
    print(Counter(dataset["train"]["final_decision"]))

    processed_ds = dataset.map(
        preprocess_pubmedqa_basic,
        remove_columns=dataset["train"].column_names,
    )
    if RUN_DATASET_INSPECTION:
        print(processed_ds["train"][0])

    tokenized_dataset = processed_ds.map(
        tokenize_for_t5,
        remove_columns=processed_ds["train"].column_names,
    )

    simple_batch_size = 2  # small for testing, increase if GPU allows
    simple_train_loader = DataLoader(
        tokenized_dataset["train"],
        batch_size=simple_batch_size,
        shuffle=True,
        collate_fn=collate_fn,
    )
    if RUN_DATASET_INSPECTION:
        print(next(iter(simple_train_loader)))

    rank = 128
    alpha = 256

    freeze_all_params(model)
    apply_lora_to_ffn(model, r=rank, alpha=alpha)

    for name, param in model.named_parameters():
        if "A.weight" in name or "B.weight" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable

    print(f"Total parameters: {total:,}")
    print(f"Trainable parameters (LoRA): {trainable:,}")
    print(f"Frozen parameters: {frozen:,}")
    print(f"Trainable %: {100 * trainable / total:.4f}%")

    model.to(device)
    simple_optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-3,
    )

    num_epochs = 3
    model.train()

    for epoch in range(num_epochs):
        running_loss = 0.0
        loop = tqdm(enumerate(simple_train_loader, 1), total=len(simple_train_loader), desc=f"Epoch {epoch+1}")

        for step, batch in loop:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            simple_optimizer.zero_grad()
            loss.backward()
            simple_optimizer.step()

            running_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        avg_loss = running_loss / len(simple_train_loader)
        print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")

    merge_lora_to_linear(model)
    model.eval()

    sample_input = "Question: Does aspirin reduce heart attack risk? Instruction: Answer in 2 sentences."
    inputs = tokenizer(sample_input, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=64)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})
{'train': ['pubid', 'question', 'context', 'long_answer', 'final_decision']}
{'pubid': Value('int32'), 'question': Value('string'), 'context': {'contexts': List(Value('string')), 'labels': List(Value('string')), 'meshes': List(Value('string')), 'reasoning_required_pred': List(Value('string')), 'reasoning_free_pred': List(Value('string'))}, 'long_answer': Value('string'), 'final_decision': Value('string')}
{'pubid': 21645374, 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?', 'context': {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occ

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'input_ids': tensor([[11860,    10,    27,     7,  9323,    61,   371,    18,   371, 13535,
             3,     9,   244,  3822,   342,  8320,    52,    12,  3613,  8985,
         10950, 19838,    58,  1193,  6327,    10,  9507,   127,    32,   221,
          9773, 13492,   509,     7,    15,    41,   371, 13535,    61,    65,
           118,  2196,    38,     3,     9,   244,  3822,   342,  8320,    52,
            12,  3613,  8985, 10950, 19838,    28,     3, 19882,  6255, 24578,
            12,    51,  5984,    41,  5668,   382,   137,    37, 22455,    19,
            24,   132,    19,    46,  1936,    95,  4914,    13,   377, 13535,
           365, 10950,   226,   447,  1124,  6980,    12,  8358, 27170, 21091,
             6, 21398,  1014,     8, 10950, 19838,    18, 14515,  1453,    13,
             3, 10791,   827,   999,     5,     3,  8656,  2116,    43,   641,
          8705,    48,   962,     6,   128,    28,  4129,    53,   772,     5,
           100,   810,     3,  8287,  

Epoch 1: 100%|██████████| 500/500 [01:27<00:00,  5.70it/s, loss=2.02] 


Epoch 1 average loss: 2.2279


Epoch 2: 100%|██████████| 500/500 [01:27<00:00,  5.74it/s, loss=2.11] 


Epoch 2 average loss: 2.0351


Epoch 3: 100%|██████████| 500/500 [01:27<00:00,  5.73it/s, loss=2.32] 


Epoch 3 average loss: 1.8021
Answer: yes. Explanation: Aspirin reduces heart attack risk.


## PubMedQA training (with short-answer accuracy eval)

In [177]:
if RUN_PUBMEDQA_TRAINING_WITH_ACCURACY:
    # ---- Dataset ----
    dataset = load_dataset("pubmed_qa", "pqa_labeled")
    if RUN_DATASET_INSPECTION:
        print(dataset)
        print(dataset["train"].features)
        print(dataset["train"][0])

    dataset = dataset.map(
        preprocess_pubmedqa_with_short_answer,
        remove_columns=dataset["train"].column_names,
    )
    if RUN_DATASET_INSPECTION:
        print(dataset["train"][0])

    split_datasets = dataset["train"].train_test_split(test_size=0.1, seed=42)

    train_data = split_datasets["train"]
    val_data = split_datasets["test"]

    if RUN_DATASET_INSPECTION:
        print(train_data)
        print(val_data)

    # Keep short answers for evaluation, then remove from features used by the model
    val_short_answers = [ex["short_answer"] for ex in val_data]
    train_data = train_data.remove_columns("short_answer")
    val_data = val_data.remove_columns("short_answer")

    if RUN_DATASET_INSPECTION:
        print(len(val_short_answers))

    train_dataset = train_data.map(tokenize_for_t5, remove_columns=train_data.column_names)
    val_dataset = val_data.map(tokenize_for_t5, remove_columns=val_data.column_names)

    # ---- Dataloaders (VRAM-friendly) ----
    # Batch size 8 with seq_len 512 will OOM on a 15GB T4; use smaller batch + grad accumulation.
    train_batch_size = 1
    val_batch_size = 1
    grad_accum_steps = 8  # effective batch ~= train_batch_size * grad_accum_steps

    train_loader = DataLoader(
        train_dataset,
        batch_size=train_batch_size,
        shuffle=True,
        collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=val_batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )
    if RUN_DATASET_INSPECTION:
        print(next(iter(train_loader)))

    # ---- LoRA config ----
    # Smaller LoRA rank keeps trainable params lower (activation memory is still the main factor).
    rank = 16
    alpha = 32

    freeze_all_params(model)
    apply_lora_to_ffn(model, r=rank, alpha=alpha)

    for name, param in model.named_parameters():
        if "A.weight" in name or "B.weight" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable

    print(f"Total parameters: {total:,}")
    print(f"Trainable parameters (LoRA): {trainable:,}")
    print(f"Frozen parameters: {frozen:,}")
    print(f"Trainable %: {100 * trainable / total:.4f}%")

    # ---- Memory optimizations ----
    def _enable_input_require_grads_for_checkpointing(_model: nn.Module) -> None:
        """
        HF gradient checkpointing uses torch.utils.checkpoint (re-entrant).
        If *all* inputs to the checkpointed functions have requires_grad=False,
        the graph can be dropped and loss won't require grad (breaking LoRA-only training).
       
        This matches the common PEFT workaround: force input-embedding outputs to require grad.
        """
        if hasattr(_model, "enable_input_require_grads"):
            _model.enable_input_require_grads()
            return
        emb = _model.get_input_embeddings() if hasattr(_model, "get_input_embeddings") else None
        if emb is None:
            return
        def _make_output_require_grad(module, inputs, output):
            if isinstance(output, torch.Tensor):
                output.requires_grad_(True)
        emb.register_forward_hook(_make_output_require_grad)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        # Gradient checkpointing reduces activation memory (important for T5).
        model.config.use_cache = False
        model.gradient_checkpointing_enable()
        _enable_input_require_grads_for_checkpointing(model)
        # TF32 can speed up matmuls on Ampere+; safe no-op elsewhere.
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    model.to(device)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-3,
    )

    use_amp = torch.cuda.is_available()
    amp_device_type = "cuda" if use_amp else "cpu"
    scaler = torch.amp.GradScaler(amp_device_type, enabled=use_amp)

    # ---- Sanity check: verify loss tracks gradients before long training ----
    _batch0 = next(iter(train_loader))
    _input_ids0 = _batch0["input_ids"].to(device)
    _attention_mask0 = _batch0["attention_mask"].to(device)
    _labels0 = _batch0["labels"].to(device)
    with torch.amp.autocast(device_type=amp_device_type, enabled=use_amp):
        _out0 = model(input_ids=_input_ids0, attention_mask=_attention_mask0, labels=_labels0)
    if not _out0.loss.requires_grad:
        raise RuntimeError(
            "Loss does not require grad. This usually means gradient checkpointing dropped the graph "
            "because inputs didn't require grads. Try disabling checkpointing or ensure input grads are enabled."
        )
    del _batch0, _input_ids0, _attention_mask0, _labels0, _out0

    num_epochs = 3
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad(set_to_none=True)

        loop = tqdm(enumerate(train_loader, 1), total=len(train_loader), desc=f"Epoch {epoch+1}")
        for step, batch in loop:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.amp.autocast(device_type=amp_device_type, enabled=use_amp):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss / grad_accum_steps

            if use_amp:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if step % grad_accum_steps == 0:
                if use_amp:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            running_loss += loss.item() * grad_accum_steps
            loop.set_postfix(loss=(loss.item() * grad_accum_steps))

        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")

        # ---- Evaluation: short-answer accuracy ----
        model.eval()
        enable_lora_forward(model)  # temporarily enable LoRA for generation

        correct = 0
        total_eval = 0
        global_idx = 0  # to index val_short_answers

        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Evaluating"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)

                outputs = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=64,
                    do_sample=False,
                )

                gen_texts = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
                for text in gen_texts:
                    match = re.search(r"Answer:\\s*(yes|no|maybe)", text, re.IGNORECASE)
                    pred = match.group(1).lower() if match else ""
                    true_label = val_short_answers[global_idx].lower()
                    if pred == true_label:
                        correct += 1
                    total_eval += 1
                    global_idx += 1

        accuracy = correct / total_eval if total_eval > 0 else 0.0
        print(f"Epoch {epoch+1} short-answer accuracy: {accuracy:.4f}")

        disable_lora_forward(model)  # restore training forward

    # ---- Generate a few samples on val set ----
    enable_lora_forward(model)
    model.eval()

    generated_texts = []
    raw_val_texts = [ex["input"] for ex in val_data]
    raw_val_labels = [ex["output"] for ex in val_data]

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Generating on val set"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=64,
                do_sample=False,
            )

            batch_texts = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
            generated_texts.extend(batch_texts)

    for i in range(5):  # first 5 examples
        print(f"Example {i+1}:")
        print(f"Question + Context + Instruction:\\n{raw_val_texts[i]}\\n")
        print(f"True Short Answer: {raw_val_labels[i]}")
        print(f"Generated Answer:\\n{generated_texts[i]}\\n")
        print("-" * 80)

    disable_lora_forward(model)

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})
{'pubid': Value('int32'), 'question': Value('string'), 'context': {'contexts': List(Value('string')), 'labels': List(Value('string')), 'meshes': List(Value('string')), 'reasoning_required_pred': List(Value('string')), 'reasoning_free_pred': List(Value('string'))}, 'long_answer': Value('string'), 'final_decision': Value('string')}
{'pubid': 21645374, 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?', 'context': {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stop

Epoch 1: 100%|██████████| 900/900 [03:04<00:00,  4.87it/s, loss=nan]


Epoch 1 average loss: nan


Evaluating: 100%|██████████| 100/100 [01:31<00:00,  1.10it/s]


Epoch 1 short-answer accuracy: 0.0000


...

Epoch 3 average loss: nan


Evaluating: 100%|██████████| 100/100 [01:31<00:00,  1.09it/s]


Epoch 3 short-answer accuracy: 0.0000


Generating on val set: 100%|██████████| 100/100 [01:31<00:00,  1.10it/s]

Example 1:
Question + Context + Instruction:\nQuestion: Is eligibility for a chemotherapy protocol a good prognostic factor for invasive bladder cancer after radical cystectomy? Context: To assess whether eligibility to an adjuvant chemotherapy protocol in itself represents a good prognostic factor after radical cystectomy for bladder cancer. Between April 1984 and May 1989, our institution entered 35 patients with invasive bladder cancer into the Swiss Group for Clinical and Epidemiological Cancer Research (SAKK) study 09/84. They were randomly assigned to either observation or three postoperative courses of cisplatin monotherapy after cystectomy. This study had a negative result. The outcome of these 35 patients (protocol group) was compared with an age- and tumor-stage-matched cohort (matched group; n = 35) who also underwent cystectomy during the same period, but were not entered into the SAKK study, as well as the remaining 57 patients treated during the study period for the same 